In [84]:
from datasets import load_dataset, ClassLabel, load_from_disk

- 加载数据集
  - 加载单个文件
  - 加载多个文件
  - 查看数据集
- API 调用（数据预处理）
  - 删除列
  - 过滤行
  - ClassLabel
  - 划分数据集、分层分组
  - 批量映射
  - tensor 格式化
- 保存数据集
  - Arrow
  - Json
  - CSV
- 集成 DataLoader

## 加载数据集

### 加载单个文件

In [85]:
dataset_dict = load_dataset('csv', data_files='../data/test.csv')
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
})


In [86]:
print(dataset_dict['train'])

Dataset({
    features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
    num_rows: 30
})


### 加载多个文件

In [87]:
dataset_dict = load_dataset('csv', data_files={
    'train': '../data/test.csv',
    'eval': '../data/test.csv',
    'test': '../data/test.csv'
})
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
    eval: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
    test: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
})


In [88]:
train_dataset = dataset_dict['train']
eval_dataset = dataset_dict['eval']
test_dataset = dataset_dict['test']
print(test_dataset)

Dataset({
    features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
    num_rows: 30
})


### 查看数据集

In [89]:
#  获取某一行（字典）
print(train_dataset[0])

{'EmployeeID': 1001, 'Name': 'John Smith', 'Department': 'Sales', 'Sales(USD)': 125000, 'PerformanceRating': 'A', 'Gender': 'M'}


In [90]:
# 获取某一列
print(test_dataset['Name'])

Column(['John Smith', 'Emma Johnson', 'Michael Brown', 'Sophia Davis', 'William Wilson', ...])


In [91]:
# 切片
print(eval_dataset[1:3])

{'EmployeeID': [1002, 1003], 'Name': ['Emma Johnson', 'Michael Brown'], 'Department': ['Marketing', 'Engineering'], 'Sales(USD)': [87600, 0], 'PerformanceRating': ['B', 'C'], 'Gender': ['F', 'M']}


In [92]:
# 访问某个字段值
print(test_dataset[1:3]['Department'])

['Marketing', 'Engineering']


## API 调用（数据预处理）

In [93]:
# 1. 删除列
dataset = train_dataset.remove_columns('Sales(USD)')
print(dataset)

Dataset({
    features: ['EmployeeID', 'Name', 'Department', 'PerformanceRating', 'Gender'],
    num_rows: 30
})


In [94]:
# 2. 过滤行
print(train_dataset['Name'])
dataset = train_dataset.filter( lambda x: x['Department'] == 'Engineering')
print(dataset['Name'])

Column(['John Smith', 'Emma Johnson', 'Michael Brown', 'Sophia Davis', 'William Wilson', ...])
Column(['Michael Brown', 'David King', 'Emily Wright', 'Matthew Lopez', 'Harper Hill', ...])


In [95]:
# 3. ClassLabel str -> int
# 3.1 定义类别映射
cl = ClassLabel(names=['Engineering', 'Marketing', 'Sales'])

# 3.2 map：字符串转int编码
dataset = train_dataset.map(lambda x: {"Department": cl.str2int(x["Department"])})

# 3.3 cast_column：给这个int列挂上ClassLabel元信息
dataset = dataset.cast_column("Department", cl)
print(dataset.features['Department'])
print(dataset['Department'])

ClassLabel(names=['Engineering', 'Marketing', 'Sales'])
Column([2, 1, 0, 2, 1, ...])


In [96]:
dataset = dataset.cast_column('Department', ClassLabel(names=['E', 'M', 'S']))
print(dataset.features['Department'])
print(dataset['Department'])

ClassLabel(names=['E', 'M', 'S'])
Column([2, 1, 0, 2, 1, ...])


In [97]:
# 4. 划分数据集
dataset = train_dataset

cl = ClassLabel(names=['Engineering', 'Marketing', 'Sales'])
gl = ClassLabel(names=['M', 'F'])

def encode_fn(x):
    return {
        "Department": cl.str2int(x["Department"]),
        "Gender": gl.str2int(x["Gender"])
    }
dataset = dataset.map(encode_fn)

dataset = dataset.cast_column('Department', cl)
dataset = dataset.cast_column('Gender', gl)

# 分层分组，这样让 2:8 的数据中，M:F 与元数据分布相似。
new_dict = dataset.train_test_split(test_size = 0.2, stratify_by_column='Gender')
print(new_dict)

DatasetDict({
    train: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 24
    })
    test: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 6
    })
})


In [98]:
# 5. 批量映射
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

In [99]:
def encode_fn(batch):
    inputs = tokenizer(
        batch['Name'],
        padding='max_length',
        max_length=8,
        truncation=True
    )
    inputs['labels'] = batch['Gender']
    return inputs

dataset = train_dataset.map(
    encode_fn,
    batched=True,
    remove_columns=['EmployeeID','Name','Department','Sales(USD)','PerformanceRating','Gender']
    )

print(dataset)
print(dataset[0:3])
print(dataset[0:3]['input_ids'])


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 30
})
{'input_ids': [[101, 2198, 3044, 102, 0, 0, 0, 0], [101, 5616, 3779, 102, 0, 0, 0, 0], [101, 2745, 2829, 102, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0]], 'labels': ['M', 'F', 'M']}
[[101, 2198, 3044, 102, 0, 0, 0, 0], [101, 5616, 3779, 102, 0, 0, 0, 0], [101, 2745, 2829, 102, 0, 0, 0, 0]]


In [100]:
# encode_fn 还可以用作数据字典的映射
a_dict = dataset_dict.map(encode_fn, batched=True, remove_columns=['EmployeeID','Name','Department','Sales(USD)','PerformanceRating','Gender'])
print(a_dict)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
    eval: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
})


## 保存数据集

### Arrow

In [101]:
# 1. 存储
a_dict.save_to_disk('data/arrow')

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

In [102]:
# 2. 导入
dataset = load_from_disk('data/arrow')
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
    eval: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 30
    })
})


In [103]:
# 3. 加载单个数据集
dataset = load_from_disk('data/arrow/train')
print(dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 30
})


### Json

In [104]:
# 1. 存储
train_dataset.to_json('data/json/train.jsonl')

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

3654

In [105]:
# 2. 载入
dataset = load_dataset('json', data_files='data/json/train.jsonl')
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
})


### CSV

In [106]:
# 1. 存储
train_dataset.to_csv('data/csv/train.csv')

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1198

In [107]:
# 2. 载入
dataset = load_dataset('csv', data_files='data/csv/train.csv')
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
        num_rows: 30
    })
})


## 集成 Dataloader

In [141]:
from torch.utils.data import DataLoader

In [142]:
dataset = train_dataset
print(dataset)
print(dataset[:5])

Dataset({
    features: ['EmployeeID', 'Name', 'Department', 'Sales(USD)', 'PerformanceRating', 'Gender'],
    num_rows: 30
})
{'EmployeeID': tensor([1001, 1002, 1003, 1004, 1005]), 'Name': ['John Smith', 'Emma Johnson', 'Michael Brown', 'Sophia Davis', 'William Wilson'], 'Department': ['Sales', 'Marketing', 'Engineering', 'Sales', 'Marketing'], 'Sales(USD)': tensor([125000,  87600,      0, 153200,  69200]), 'PerformanceRating': ['A', 'B', 'C', 'A', 'B'], 'Gender': ['M', 'F', 'M', 'F', 'M']}


In [143]:
def encode_fn(batch):
    inputs = tokenizer(
        batch['Name'],
        padding='max_length',
        max_length=8,
        truncation=True
    )
    return inputs

dataset = dataset.map(
    encode_fn,
    batched=True,
    remove_columns=['EmployeeID','Name','Department','Sales(USD)','PerformanceRating','Gender']
    )

dataset.set_format(type='torch')
print(dataset[:5])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

{'input_ids': tensor([[ 101, 2198, 3044,  102,    0,    0,    0,    0],
        [ 101, 5616, 3779,  102,    0,    0,    0,    0],
        [ 101, 2745, 2829,  102,    0,    0,    0,    0],
        [ 101, 9665, 4482,  102,    0,    0,    0,    0],
        [ 101, 2520, 4267,  102,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0]])}


In [144]:
dataloader = DataLoader(dataset, shuffle=True, batch_size=32)

In [145]:
for batch in dataloader:
    for k,v in batch.items():
        print(k, ' -> ', v.shape)
    break

input_ids  ->  torch.Size([30, 8])
token_type_ids  ->  torch.Size([30, 8])
attention_mask  ->  torch.Size([30, 8])
